In [3]:
import numpy as np
import pandas as pd
from transformers import AutoImageProcessor, AutoModel
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import torch.nn.functional as F
import os
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
import cv2
from tqdm.notebook import tqdm

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
print('device:',device)

device: cuda


In [19]:
if Path('/kaggle/input').exists():
    model_path = '/kaggle/input/models/metaresearch/dinov2/pytorch/base/1/'
    dataset_path = '/kaggle/input/datasets/xiaose/cityscapes/Cityspaces/images'
else:
    notebook_path = Path.cwd()
    root_path = notebook_path.parent.parent 
    cityscapes_dataset_path = root_path / 'data' / 'Cityspaces' / 'images'
    model_path = root_path / 'model'
    comma_dataset_path = root_path / 'data' / 'comma2k19'    
    comma_embeddings_path = root_path / 'data' / "comma_dino_embeddings_1"
print(f'Comma path: {comma_dataset_path}')

Comma path: /home/hxastur/vscode-projects/autonomous-driving/data/comma2k19


```
Dataset_chunk_n
|
+-- route_id (dongle_id|start_time)
    |
    +-- segment_number
        |
        +-- preview.png (first frame video)
        +-- raw_log.bz2 (raw capnp log, can be read with openpilot-tools: logreader)
        +-- video.hevc (video file, can be read with openpilot-tools: framereader)
        +-- processed_log/ (processed logs as numpy arrays, see format for details)
        +-- global_pos/ (global poses of camera as numpy arrays, see format for details)
```

In [6]:
def build_index(comma_dataset_path):
    chunks = [x.name for x in Path(comma_dataset_path).iterdir() if x.is_dir()]
    samples = []
    for chunk in chunks:
        chunk_path = comma_dataset_path / chunk
        routes = [x for x in Path(chunk_path).iterdir() if x.is_dir()]
        for route in routes: 
            segments = [x for x in Path(route).iterdir() if x.is_dir()]
            for segment in segments: 
                video_path = segment / 'video.hevc'
                
                steering_angle_path = segment / 'processed_log' / 'CAN' / 'steering_angle'
                steering_angle_value=np.load(steering_angle_path / 'value')
                steering_angle_t=np.load(steering_angle_path / 't')

                frame_times_path = segment / 'global_pose' / 'frame_times'
                frame_times = np.load(frame_times_path)

                steering_angle_idx = np.searchsorted(
                    steering_angle_t,
                    frame_times
                )
                for i,frame_time in enumerate(frame_times):
                    idx = steering_angle_idx[i]
                    if idx == 0:
                        nearest = 0
                    elif idx == len(steering_angle_t):
                        nearest = len(steering_angle_t) - 1
                    else:
                        left = idx - 1
                        right = idx
                    
                        if abs(frame_time - steering_angle_t[left]) < abs(frame_time - steering_angle_t[right]):
                            nearest = left
                        else:
                            nearest = right
                    
                    sample = {
                    'video_path': video_path,
                    'chunk':chunk,
                    'route':route.name,
                    'segment':segment.name,
                    'frame_idx':i,
                    'frame_time':frame_time,
                    'steering_angle': steering_angle_value[nearest]
                    }
                    samples.append(sample)
    return samples
samples = build_index(comma_dataset_path=comma_dataset_path)
samples[1]

{'video_path': PosixPath('/home/hxastur/vscode-projects/autonomous-driving/data/comma2k19/Chunk_1/b0c9d2329ad1606b|2018-07-30--13-03-07/22/video.hevc'),
 'chunk': 'Chunk_1',
 'route': 'b0c9d2329ad1606b|2018-07-30--13-03-07',
 'segment': '22',
 'frame_idx': 1,
 'frame_time': np.float64(11215.100412),
 'steering_angle': np.float64(3.5)}

In [7]:
def read_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    status, frame = cap.read()
    cap.release()
    return frame    

In [14]:
dino_model = AutoModel.from_pretrained(model_path, device_map="cuda")
dino_processor = AutoImageProcessor.from_pretrained(model_path)

from collections import defaultdict

def create_embeddings(samples, device):
    # 1. Группируем индексы по видео
    video_to_indices = defaultdict(list)
    for i, sample in enumerate(samples):
        video_to_indices[sample['video_path']].append((i, sample['frame_idx']))
    
    # Массив для результатов (сохраняем порядок исходных samples)
    embeddings = [None] * len(samples)
    
    # 2. Обрабатываем каждое видео отдельно
    for video_path, idx_list in tqdm(video_to_indices.items(), desc="Videos"):
        # Сортируем запросы по frame_idx для последовательного чтения
        idx_list.sort(key=lambda x: x[1])
        
        cap = cv2.VideoCapture(video_path)
        current_frame = 0
        
        for original_pos, target_frame in idx_list:
            # Перемещаемся вперёд (или назад, если неупорядочено — но мы отсортировали)
            if target_frame < current_frame:
                # Если просим кадр раньше текущего — переоткрываем или сбрасываем
                # Проще переоткрыть, но лучше избегать такой ситуации (сортировка по возрастанию гарантирует target_frame >= current_frame)
                cap.release()
                cap = cv2.VideoCapture(video_path)
                current_frame = 0
            
            # Пропускаем нужное количество кадров
            while current_frame < target_frame:
                ret, _ = cap.read()
                if not ret:
                    break
                current_frame += 1
            
            ret, frame = cap.read()
            if ret:
                current_frame += 1
                # Обработка кадра
                pixel_values = dino_processor(images=frame, return_tensors='pt')['pixel_values'].to(device)
                with torch.no_grad():
                    last_hidden = dino_model(pixel_values).last_hidden_state[:, 0]
                embeddings[original_pos] = last_hidden.detach().cpu().numpy()
            else:
                # Ошибка чтения кадра, можно заполнить нулями
                embeddings[original_pos] = np.zeros(dino_model.config.hidden_size)
        
        cap.release()
    
    # Убираем None (если всё ок, их не должно быть)
    return np.stack(embeddings)

def save_embeddings(embeddings, comma_embeddings_path):
    np.save(comma_embeddings_path, embeddings)

def load_embeddings(comma_embeddings_path):
    embeddings = np.load(comma_embeddings_path)
    return embeddings

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

In [22]:
# embeddings = create_embeddings(samples, device)
# save_embeddings(embeddings, comma_embeddings_path)
embeddings = load_embeddings(str(comma_embeddings_path)+'.npy')

In [23]:
class CommaDataset(Dataset):
    def __init__(self, samples, embeddings):
        self.samples = samples
        self.embeddings = embeddings
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        # frame = read_frame(sample['video_path'], sample['frame_idx'])
        steering = sample['steering_angle']
        # pixel_values = processor(images=frame, return_tensors='pt')['pixel_values'].squeeze(0)
        embedding = embeddings[idx]
        return embedding, steering

In [24]:
dataset = CommaDataset(samples,embeddings)
dataiter = iter(dataset)
next(dataiter)[0].shape

(1, 768)

In [25]:
class CommaBCNet(nn.Module):
    def __init__(self,params1 = 256):
        super().__init__()
        
        # self.dino = AutoModel.from_pretrained(model_path)
        # for p in self.dino.parameters():
        #     p.requires_grad = False

        self.head = nn.Sequential( 
            nn.Linear(768, params1),
            nn.ReLU(),
            nn.Linear(params1, 1)
        )
    def forward(self, input):
        # dino_outputs = self.dino(pixel_values=input)
        # embedding = dino_outputs.last_hidden_state[:, 0]
        output = self.head(input)
        return output.squeeze(1)

In [26]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(dataset, batch_size = 64, shuffle=True)
net = CommaBCNet().to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [27]:
running_loss = 0.
last_loss = 0.
for i,data in tqdm(enumerate(train_dataloader),total=len(train_dataloader)):
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)
    optimizer.zero_grad()
    outputs = net(inputs)
    loss = loss_fn(outputs, labels)
    loss.backward()

    optimizer.step()

    running_loss += loss.item()
    if i % 1000 == 999:
        last_loss = running_loss / 1000
        print(f'  batch {i + 1} loss: {last_loss}')
        # tb_x = epoch_index * len(training_loader) + i + 1
        # tb_writer.add_scalar('Loss/train', last_loss, tb_x)
        running_loss = 0.

  0%|          | 0/3513 [00:00<?, ?it/s]

/home/hxastur/vscode-projects/autonomous-driving/.venv/lib/python3.11/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


  batch 1000 loss: 265.5341547532257
  batch 2000 loss: 292.07875402069743
  batch 3000 loss: 254.13146482307033


/home/hxastur/vscode-projects/autonomous-driving/.venv/lib/python3.11/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([15])) that is different to the input size (torch.Size([15, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
